In [ ]:
library(Seurat)
library(readr)
library(dplyr)
library(ggplot2)
library(pheatmap)
library(tidyr) 
library(patchwork)
library(tidyverse)
library(ggrepel)

In [ ]:
## 鲸豚整合

In [ ]:
data <- readRDS("data/processed/excitatory_neuron_file")
data <- subset(data, subset = annotation != "Other")
data

In [ ]:
gene1_expr <- GetAssayData(data, assay = "RNA", slot = "data")["BDNF", ]
gene2_expr <- GetAssayData(data, assay = "RNA", slot = "data")["ADCYAP1", ]

# 找到两个基因同时表达的细胞（假设表达值大于即为表达）
expressed_cells <- (gene1_expr > 1) & (gene2_expr > 1)

# 计算两个基因的表达量之和，仅在同时表达的细胞中
sum_expr <- rep(0, length(gene1_expr)) # 初始化为0
sum_expr[expressed_cells] <- gene1_expr[expressed_cells] + gene2_expr[expressed_cells]

data[["RNA"]]@data <- rbind(data[["RNA"]]@data, sum_expr)
rownames(data[["RNA"]]@data)[nrow(data[["RNA"]]@data)] <- "BDNF_ADCYAP1"
head(data[["RNA"]]@data["BDNF_ADCYAP1", ])

In [ ]:
FeaturePlot(data, features = "BDNF_ADCYAP1", reduction = "umap", pt.size = 0.1)

In [ ]:
genes_to_plot <- c("BDNF", "ADCYAP1", "BDNF_ADCYAP1","PLAT")
dotplot <- DotPlot(data, features = genes_to_plot, group.by = "annotation", 
                   dot.scale = 6) + # 移除DotPlot的cols参数
  # 控制点大小图例范围（0-100%）
  scale_size_continuous(
    limits = c(0, 100), 
    breaks = c(0, 25, 50, 75, 100),  # 自定义刻度
    labels = c("0%", "25%", "50%", "75%", "100%"),  # 添加百分比符号
    name = "Percent Expressed"  # 图例标题
  ) +
  # 控制颜色图例范围（-1到2）
   scale_color_gradient(
    low = "#1F66AC",   # 低值：蓝色
    high = "#B2192B",  # 高值：红色
    limits = c(-1, 3), 
    breaks = c(-1, 0, 1, 2, 3),
    name = "Expression Level"
  ) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    plot.title = element_blank(),
    axis.title.x = element_blank(),
    axis.title.y = element_blank(),
    # 图例样式优化
    legend.position = "right",  # 图例位置[2](@ref)
    legend.key.height = unit(0.5, "cm"),  # 颜色图例高度[6](@ref)
    legend.box.spacing = unit(0.2, "cm")   # 图例间距
  )

ggsave("results/figures/figure_panel_file", plot = dotplot, width = 5, height = 8)
ggsave("results/figures/figure_panel_file", plot = dotplot, width = 5, height = 8)

In [ ]:
## 牛

In [ ]:
datab <- readRDS("data/processed/cortex_analysis_file")
datab$annotation <- gsub("/", "_", datab$annotation)
datab

In [ ]:
DimPlot(datab, label =TRUE, pt.size = 0.1, group.by = "annotation")

In [ ]:
cluster_colours <- c(
  Ex1          = "#E69F00",
  Ex2          = "#2D5F9E",
  Ex3          = "#006D4D",
  Ex4          = "#004B87",
  Ex5          = "#D55E00",
  Ex6          = "#CC79A7",
  Ex7          = "#332288",
  In1          = "#88CCEE",
  In2          = "#7E1F86",
  Oligo        = "#117733",
  Oligo_Astro  = "#661100",   # 原来 Oligo/Astro
  Oligo_Opc    = "#999933",   # 原来 Oligo/Opc
  Opc          = "#DDCC77",
  `Oligo(TMCC3)` = "#44AA99",
  Astro = "#6baed6",
  Endo = "#08519c"
)

p <- DimPlot(datab, label = FALSE, pt.size = 0.1, group.by = "annotation", cols = cluster_colours) +
  theme_void() +
  theme(legend.position = "none",  # 去除图例
        axis.text = element_blank(),  # 去除坐标轴文字
        axis.ticks = element_blank(),  # 去除坐标轴刻度
        plot.title = element_blank(),  # 去除标题
        plot.subtitle = element_blank(),  # 去除副标题
        plot.caption = element_blank())  # 去除图注

ggsave("results/figures/figure_panel_file", plot = p, device = "png", width = 8, height = 8)
ggsave("results/figures/figure_panel_file", plot = p, device = "pdf", width = 8, height = 8)

In [ ]:
# 第二个牛数据

In [ ]:
datab2 <- readRDS("data/processed/cortex_analysis_file")
datab2$annotation <- gsub("/", "_", datab2$annotation)
datab2

In [ ]:
DimPlot(datab2, label =TRUE, pt.size = 0.1, group.by = "annotation")

In [ ]:
cluster_colours <- c(
  Ex1          = "#E69F00",
  Ex2          = "#2D5F9E",
  Ex3          = "#006D4D",
  Ex4          = "#004B87",
  Ex5          = "#D55E00",
  Ex6          = "#CC79A7",
  Micro        = "#332288",
  In1          = "#88CCEE",
  In2          = "#7E1F86",
  In3          = "#5C7D9A",   # 低饱和冷灰蓝
  In4          = "#8A7CA9",   # 低饱和灰紫
  Oligo        = "#117733",
  Oligo_Astro  = "#661100",   # 原来 Oligo/Astro
  Oligo_Opc    = "#999933",   # 原来 Oligo/Opc
  Opc          = "#DDCC77",
  `Oligo(TMCC3)` = "#44AA99"
)

p <- DimPlot(datab2, label = FALSE, pt.size = 0.1, group.by = "annotation", cols = cluster_colours) +
  theme_void() +
  theme(legend.position = "none",  # 去除图例
        axis.text = element_blank(),  # 去除坐标轴文字
        axis.ticks = element_blank(),  # 去除坐标轴刻度
        plot.title = element_blank(),  # 去除标题
        plot.subtitle = element_blank(),  # 去除副标题
        plot.caption = element_blank())  # 去除图注

ggsave("results/figures/figure_panel_file", plot = p, device = "png", width = 8, height = 8)
ggsave("results/figures/figure_panel_file", plot = p, device = "pdf", width = 8, height = 8)

In [ ]:
## 计算牛共表达(选用第一个数据集

In [ ]:
datab <- readRDS("data/processed/cortex_analysis_file")
datab

In [ ]:
gene1_expr <- GetAssayData(datab, assay = "RNA", slot = "data")["BDNF", ]
gene2_expr <- GetAssayData(datab, assay = "RNA", slot = "data")["ADCYAP1", ]

# 找到两个基因同时表达的细胞（假设表达值大于即为表达）
expressed_cells <- (gene1_expr > 1) & (gene2_expr > 1)

# 计算两个基因的表达量之和，仅在同时表达的细胞中
sum_expr <- rep(0, length(gene1_expr)) # 初始化为0
sum_expr[expressed_cells] <- gene1_expr[expressed_cells] + gene2_expr[expressed_cells]

datab[["RNA"]]@data <- rbind(datab[["RNA"]]@data, sum_expr)
rownames(datab[["RNA"]]@data)[nrow(datab[["RNA"]]@data)] <- "BDNF_ADCYAP1"
head(datab[["RNA"]]@data["BDNF_ADCYAP1", ])

In [ ]:
FeaturePlot(datab, features = "BDNF_ADCYAP1", reduction = "umap", pt.size = 0.1)

In [ ]:
genes_to_plot <- c("BDNF", "ADCYAP1", "BDNF_ADCYAP1","PLAT")
dotplot <- DotPlot(datab, features = genes_to_plot, group.by = "annotation", 
                   dot.scale = 6) + # 移除DotPlot的cols参数
  # 控制点大小图例范围（0-100%）
  scale_size_continuous(
    limits = c(0, 100), 
    breaks = c(0, 25, 50, 75, 100),  # 自定义刻度
    labels = c("0%", "25%", "50%", "75%", "100%"),  # 添加百分比符号
    name = "Percent Expressed"  # 图例标题
  ) +
  # 控制颜色图例范围（-1到2）
   scale_color_gradient(
    low = "#1F66AC",   # 低值：蓝色
    high = "#B2192B",  # 高值：红色
    limits = c(-1, 3), 
    breaks = c(-1, 0, 1, 2, 3),
    name = "Expression Level"
  ) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    plot.title = element_blank(),
    axis.title.x = element_blank(),
    axis.title.y = element_blank(),
    # 图例样式优化
    legend.position = "right",  # 图例位置[2](@ref)
    legend.key.height = unit(0.5, "cm"),  # 颜色图例高度[6](@ref)
    legend.box.spacing = unit(0.2, "cm")   # 图例间距
  )
ggsave("results/figures/figure_panel_file", plot = dotplot, width = 4, height = 8)
ggsave("results/figures/figure_panel_file", plot = dotplot, width = 4, height = 8)

In [ ]:
## mouse

In [ ]:
datamu <- readRDS("data/processed/cortex_analysis_file")
datamu

In [ ]:
gene1_expr <- GetAssayData(datamu, assay = "RNA", slot = "data")["BDNF", ]
gene2_expr <- GetAssayData(datamu, assay = "RNA", slot = "data")["ADCYAP1", ]

# 找到两个基因同时表达的细胞（假设表达值大于即为表达）
expressed_cells <- (gene1_expr > 1) & (gene2_expr > 1)

# 计算两个基因的表达量之和，仅在同时表达的细胞中
sum_expr <- rep(0, length(gene1_expr)) # 初始化为0
sum_expr[expressed_cells] <- gene1_expr[expressed_cells] + gene2_expr[expressed_cells]

datamu[["RNA"]]@data <- rbind(datamu[["RNA"]]@data, sum_expr)
rownames(datamu[["RNA"]]@data)[nrow(datamu[["RNA"]]@data)] <- "BDNF_ADCYAP1"
head(datamu[["RNA"]]@data["BDNF_ADCYAP1", ])

In [ ]:
FeaturePlot(datamu, features = "BDNF_ADCYAP1", reduction = "umap", pt.size = 0.1)

In [ ]:
genes_to_plot <- c("BDNF", "ADCYAP1", "BDNF_ADCYAP1","PLAT")
dotplot <- DotPlot(datamu, features = genes_to_plot, group.by = "Layer", 
                   dot.scale = 6) + # 移除DotPlot的cols参数
  # 控制点大小图例范围（0-100%）
  scale_size_continuous(
    limits = c(0, 100), 
    breaks = c(0, 25, 50, 75, 100),  # 自定义刻度
    labels = c("0%", "25%", "50%", "75%", "100%"),  # 添加百分比符号
    name = "Percent Expressed"  # 图例标题
  ) +
  # 控制颜色图例范围（-1到2）
   scale_color_gradient(
    low = "#1F66AC",   # 低值：蓝色
    high = "#B2192B",  # 高值：红色
    limits = c(-1, 3), 
    breaks = c(-1, 0, 1, 2, 3),
    name = "Expression Level"
  ) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    plot.title = element_blank(),
    axis.title.x = element_blank(),
    axis.title.y = element_blank(),
    # 图例样式优化
    legend.position = "right",  # 图例位置[2](@ref)
    legend.key.height = unit(0.5, "cm"),  # 颜色图例高度[6](@ref)
    legend.box.spacing = unit(0.2, "cm")   # 图例间距
  )

ggsave("results/figures/figure_panel_file", plot = dotplot, width = 4, height = 8)
ggsave("results/figures/figure_panel_file", plot = dotplot, width = 4, height = 8)

In [ ]:
## human

In [ ]:
datahm <- readRDS("data/processed/cortex_analysis_file")
datahm

In [ ]:
gene1_expr <- GetAssayData(datahm, assay = "RNA", slot = "data")["BDNF", ]
gene2_expr <- GetAssayData(datahm, assay = "RNA", slot = "data")["ADCYAP1", ]

# 找到两个基因同时表达的细胞（假设表达值大于即为表达）
expressed_cells <- (gene1_expr > 1) & (gene2_expr > 1)

# 计算两个基因的表达量之和，仅在同时表达的细胞中
sum_expr <- rep(0, length(gene1_expr)) # 初始化为0
sum_expr[expressed_cells] <- gene1_expr[expressed_cells] + gene2_expr[expressed_cells]

datahm[["RNA"]]@data <- rbind(datahm[["RNA"]]@data, sum_expr)
rownames(datahm[["RNA"]]@data)[nrow(datahm[["RNA"]]@data)] <- "BDNF_ADCYAP1"
head(datahm[["RNA"]]@data["BDNF_ADCYAP1", ])

In [ ]:
FeaturePlot(datahm, features = "BDNF_ADCYAP1", reduction = "umap", pt.size = 0.1)

In [ ]:
genes_to_plot <- c("BDNF", "ADCYAP1", "BDNF_ADCYAP1","PLAT")
dotplot <- DotPlot(datahm, features = genes_to_plot, group.by = "Layer", 
                   dot.scale = 6) + # 移除DotPlot的cols参数
  # 控制点大小图例范围（0-100%）
  scale_size_continuous(
    limits = c(0, 100), 
    breaks = c(0, 25, 50, 75, 100),  # 自定义刻度
    labels = c("0%", "25%", "50%", "75%", "100%"),  # 添加百分比符号
    name = "Percent Expressed"  # 图例标题
  ) +
  # 控制颜色图例范围（-1到2）
   scale_color_gradient(
    low = "#1F66AC",   # 低值：蓝色
    high = "#B2192B",  # 高值：红色
    limits = c(-1, 3), 
    breaks = c(-1, 0, 1, 2, 3),
    name = "Expression Level"
  ) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    plot.title = element_blank(),
    axis.title.x = element_blank(),
    axis.title.y = element_blank(),
    # 图例样式优化
    legend.position = "right",  # 图例位置[2](@ref)
    legend.key.height = unit(0.5, "cm"),  # 颜色图例高度[6](@ref)
    legend.box.spacing = unit(0.2, "cm")   # 图例间距
  )

ggsave("results/figures/figure_panel_file", plot = dotplot, width = 4, height = 8)
ggsave("results/figures/figure_panel_file", plot = dotplot, width = 4, height = 8)